In [49]:
import numpy as np
import pandas as pd
import yfinance as yf
import datetime as dt
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')


In [50]:

# Descarga de datos
tickers = ["JPM","^MXX","PLTR","V"]
weights = np.array([0.26,0.24,0.20,0.30])
n_sim=1000
n_days=252
n_assets=len(tickers)
end = dt.datetime.today()
start = end - dt.timedelta(days=365*1)

data = yf.download(tickers,start=start,end=end)["Close"]
returns = data.pct_change().dropna()


[*********************100%***********************]  4 of 4 completed


In [51]:
lambda_ = 0.94

ewma_var = returns.var().copy()
ewma_vol_series = []

for t in range(len(returns)):
    r_t = returns.iloc[t]
    ewma_var = lambda_ * ewma_var + (1 - lambda_) * (r_t ** 2)
    ewma_vol_series.append(np.sqrt(ewma_var))

ewma_vol_df = pd.DataFrame(ewma_vol_series, 
                           index=returns.index, 
                           columns=returns.columns)

In [52]:
corr_matrix = returns.corr()
L_corr = np.linalg.cholesky(corr_matrix)

In [53]:
sigma_t=ewma_vol_df.iloc[-1].values
portfolio_paths=np.zeros((n_sim,n_days))

for s in range(n_sim):

    sigma = sigma_t.copy()
    prices_sim = np.ones(n_assets)

    for t in range(n_days):

        z = np.random.normal(size=n_assets)
        correlated = L_corr @ z

        r = sigma * correlated

        prices_sim = prices_sim * (1 + r)

        port_ret = np.dot(weights, r)

        if t == 0:
            portfolio_paths[s, t] = 1 + port_ret
        else:
            portfolio_paths[s, t] = portfolio_paths[s, t-1] * (1 + port_ret)

        sigma = np.sqrt(lambda_ * sigma**2 + (1 - lambda_) * (r**2))

In [54]:
final_values = portfolio_paths[:,-1]

var_95 = np.percentile(final_values,5)
es_95 = final_values[final_values<=var_95].mean()

print("VaR 95%:",1-var_95)
print("Expected Shortfall 95%:",1-es_95)

VaR 95%: 0.3402054640714718
Expected Shortfall 95%: 0.42175357003400304


In [ ]:
tickers = ["JPM", "^MXX", "PLTR", "V"]
weights = np.array([0.26, 0.24, 0.20, 0.30])
n_sim = 1000
n_days = 252
n_assets = len(tickers)
years_list = [1, 3, 5, 10]
lambdas_list = [0.94, 0.8]
end = dt.datetime.today()
start_max = end - dt.timedelta(days=365 * 10)
full_data = yf.download(tickers, start=start_max, end=end)["Close"]

results = []

for yrs in years_list:
    start_date = end - dt.timedelta(days=365 * yrs)
    data = full_data[full_data.index >= start_date]
    returns = data.pct_change().dropna()
    
    corr_matrix = returns.corr()
    L_corr = np.linalg.cholesky(corr_matrix)
    
    for l_val in lambdas_list:
        ewma_var = returns.var().copy()
        for t in range(len(returns)):
            r_t = returns.iloc[t]
            ewma_var = l_val * ewma_var + (1 - l_val) * (r_t ** 2)
        
        sigma_t = np.sqrt(ewma_var).values
        portfolio_paths = np.ones((n_sim, n_days))
        for s in range(n_sim):
            sigma = sigma_t.copy()
            current_port_val = 1.0
            
            for t in range(n_days):
                z = np.random.normal(size=n_assets)
                correlated = L_corr @ z
                r = sigma * correlated
                
                port_ret = np.dot(weights, r)
                current_port_val *= (1 + port_ret)
                portfolio_paths[s, t] = current_port_val
                sigma = np.sqrt(l_val * sigma**2 + (1 - l_val) * (r**2))
        final_values = portfolio_paths[:, -1]
        var_95 = np.percentile(final_values, 5)
        es_95 = final_values[final_values <= var_95].mean()
        results.append({
            "Años": yrs,
            "Lambda": l_val,
            "VaR 95%": 1 - var_95,
            "ES 95%": 1 - es_95
        })
        print(f"Completado: {yrs} años con Lambda {l_val}")

df_results = pd.DataFrame(results)


[*********************100%***********************]  4 of 4 completed


Completado: 1 años con Lambda 0.94
Completado: 1 años con Lambda 0.8
Completado: 3 años con Lambda 0.94
Completado: 3 años con Lambda 0.8
Completado: 5 años con Lambda 0.94
Completado: 5 años con Lambda 0.8
Completado: 10 años con Lambda 0.94
Completado: 10 años con Lambda 0.8


In [56]:

stocks = ["JPM", "^MXX", "PLTR", "V"]
weights = np.array([0.26, 0.24, 0.20, 0.30])
num_sim = 5000
num_days = 252
k = len(stocks)
years_list = [1, 3, 5, 10]

end = dt.datetime.now()
start_max = end - dt.timedelta(days=365 * 10)
full_prices = yf.download(stocks, start=start_max, end=end, progress=False)["Close"]

mgb_results = []

for yrs in years_list:
    start_date = end - dt.timedelta(days=365 * yrs)
    prices = full_prices[full_prices.index >= start_date]
    returns = prices.pct_change().dropna()
    
    mean_returns = returns.mean()
    cov_matrix = returns.cov()
    L = np.linalg.cholesky(cov_matrix)
    
    portfolio_paths = np.zeros((num_days, num_sim))
    
    for m in range(num_sim):
        Z = np.random.normal(size=(num_days, k))
        correlated = Z @ L.T
        daily_returns = correlated + mean_returns.values
        port_daily = daily_returns @ weights
        portfolio_paths[:, m] = np.cumprod(1 + port_daily)

    final_values = portfolio_paths[-1]
    var_95 = np.percentile(final_values, 5)
    es_95 = final_values[final_values <= var_95].mean()
    
    mgb_results.append({
        "Años": yrs,
        "VaR 95%": 1 - var_95,
        "ES 95%": 1 - es_95
    })
df_mgb_final = pd.DataFrame(mgb_results)

In [57]:
print("---------EWMA -------")

print(df_results)
print("---------Movimiento Geometrico Browniano -------")
print(df_mgb_final)

---------EWMA -------
   Años  Lambda   VaR 95%    ES 95%
0     1    0.94  0.308514  0.375241
1     1    0.80  0.174944  0.257290
2     3    0.94  0.292052  0.362280
3     3    0.80  0.159234  0.263125
4     5    0.94  0.315552  0.382828
5     5    0.80  0.177207  0.279300
6    10    0.94  0.300246  0.382397
7    10    0.80  0.179290  0.260367
---------Movimiento Geometrico Browniano -------
   Años   VaR 95%    ES 95%
0     1  0.149083  0.230745
1     3 -0.028065  0.053527
2     5  0.162622  0.241392
3    10  0.095402  0.174168
